# 01 — Data exploration and currency-pair construction

This notebook reproduces the data-preparation steps from the final notebook. It uses `daily_forex_rates.csv`, an alternative dataset to the Bloomberg data in the reference paper.

**Input:** place `daily_forex_rates.csv` in the project's `data/` folder.

In [2]:

from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("/Users/namithar/ML-Projects/FX-Forecasting")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fx_utils import load_forex_data, build_pair_rates, PAIRS

csv_path = PROJECT_ROOT / "data" / "daily_forex_rates.csv"

forex_df = load_forex_data(csv_path)

print("Shape:", forex_df.shape)
display(forex_df.head())

Shape: (505919, 5)


,currency,base_currency,currency_name,exchange_rate,date
0,MXV,EUR,NaN,2.330891,2026-10-11
1,GIP,EUR,Gibraltar Pound,0.846731,2026-10-11
2,HTG,EUR,Haitian Gourde,146.729800,2026-10-11
3,HRK,EUR,Croatian Kuna,7.535656,2026-10-11
4,HNL,EUR,Honduran Lempira,30.076760,2026-10-11


In [3]:
print("Columns:", forex_df.columns.tolist())
print("\nData types:\n", forex_df.dtypes)
print("\nDate range:", forex_df["date"].min(), "to", forex_df["date"].max())
print("\nUnique currencies:", forex_df["currency"].nunique())
print("Unique base currencies:", forex_df["base_currency"].nunique())
print("\nBase currency counts:\n", forex_df["base_currency"].value_counts().head())
print("\nMissing values:\n", forex_df.isna().sum())
print("\nDuplicate rows:", forex_df.duplicated().sum())

Columns: ['currency', 'base_currency', 'currency_name', 'exchange_rate', 'date']

Data types:
 currency                    str
base_currency               str
currency_name               str
exchange_rate           float64
date             datetime64[us]
dtype: object

Date range: 2004-08-30 00:00:00 to 2026-10-11 00:00:00

Unique currencies: 176
Unique base currencies: 1

Base currency counts:
 base_currency
EUR    505919
Name: count, dtype: int64

Missing values:
 currency            0
base_currency       0
currency_name    2923
exchange_rate       0
date                0
dtype: int64

Duplicate rows: 0


In [4]:
target = forex_df[
    forex_df["base_currency"].eq("EUR")
    & forex_df["currency"].isin(["GBP", "USD", "JPY"])
].copy()
print("Duplicate date-currency pairs:", target.duplicated(["date", "currency"]).sum())

rates_wide = target.pivot(index="date", columns="currency", values="exchange_rate").sort_index()
print("Wide table shape:", rates_wide.shape)
print("Missing selected currency rates:\n", rates_wide[["GBP", "USD", "JPY"]].isna().sum())
display(rates_wide.head())

Duplicate date-currency pairs: 0
Wide table shape: (6077, 3)
Missing selected currency rates:
 currency
GBP     3
USD    19
JPY     1
dtype: int64


currency,GBP,JPY,USD
date,,,
2004-08-31,NaN,132.85,NaN
2004-09-01,NaN,133.35,NaN
2004-09-02,NaN,133.22,NaN
2004-09-03,0.6779,133.36,NaN
2004-09-06,0.6778,132.83,NaN


In [5]:
pair_df = build_pair_rates(forex_df)
print("Pair dataset shape:", pair_df.shape)
print("Date range:", pair_df.index.min(), "to", pair_df.index.max())
print("Duplicate dates:", pair_df.index.duplicated().sum())
print("Missing values:\n", pair_df.isna().sum())
display(pair_df.head())

Pair dataset shape: (6057, 3)
Date range: 2004-09-27 00:00:00 to 2026-10-11 00:00:00
Duplicate dates: 0
Missing values:
 currency
GBP_USD    0
USD_EUR    0
USD_JPY    0
dtype: int64


currency,GBP_USD,USD_EUR,USD_JPY
date,,,
2004-09-27,1.809419,0.813339,111.240342
2004-09-28,1.814875,0.811491,111.344640
2004-09-29,1.800935,0.810964,110.785824
2004-09-30,1.812628,0.804441,110.047462
2004-10-01,1.798637,0.805997,110.373176


In [ ]:
pair_df.plot(subplots=True, figsize=(12, 8), title="Daily exchange rates")
plt.tight_layout()
plt.show()

## What this notebook establishes

- The raw source is long-form: one row per date and currency.
- With EUR as the base, the derived pairs are `GBP_USD = USD / GBP`, `USD_EUR = 1 / USD`, and `USD_JPY = JPY / USD`.
- The common complete-date dataset should have approximately 6,057 rows from 2004-09-27 through 2026-10-11, matching the final notebook's observed output.
- This dataset does not match the paper's original Bloomberg data or 2001–2018 period; describe the work as an alternative-data reproduction attempt.